# Chương 2: MLP bằng Keras (`Sequential`, `fit()`)
Cùng kiến trúc, khởi tạo He, SGD thuần, tốc độ học, epoch, kích thước lô với hai cài đặt còn lại (xem `CFG` trong `ch2_run.py`). Kernel: **Python (TF 2.21)** (môi trường Keras).
Notebook chạy nhanh một bộ dữ liệu một hạt giống; muốn chạy đủ 3 bộ x 3 hạt giống dùng `python ch2_run.py keras`.

In [1]:
import os, sys, inspect
sys.path.insert(0, os.path.abspath(".."))
os.chdir(os.path.abspath(".."))
import numpy as np
import ch2_run as C

In [2]:
print(inspect.getsource(C.run_keras)) if "keras" != "numpy" else print(inspect.getsource(__import__("dl_scratch").MLPScratch))

def run_keras(name, D, c, seed):
    os.environ["KERAS_BACKEND"] = "tensorflow"; os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
    import keras
    from keras import layers
    keras.utils.set_random_seed(seed)
    acts = ["relu"] * (len(c["sizes"]) - 2) + [{"binary": "sigmoid", "regression": None, "multiclass": "softmax"}[c["task"]]]
    net = keras.Sequential([keras.Input((c["sizes"][0],))] + [layers.Dense(b, activation=a, kernel_initializer="he_normal", bias_initializer="zeros") for b, a in zip(c["sizes"][1:], acts)])
    loss = {"binary": "binary_crossentropy", "regression": lambda t, p: 0.5 * keras.ops.mean(keras.ops.square(p - t)), "multiclass": "sparse_categorical_crossentropy"}[c["task"]]
    net.compile(optimizer=keras.optimizers.SGD(c["lr"]), loss=loss)
    bs = c["batch"] or len(D["X_tr"])
    t0 = time.perf_counter(); h = net.fit(D["X_tr"].astype("float32"), D["y_tr"].astype("float32" if c["task"] != "multiclass" else "int32"), batch_size=bs, epochs=c["epochs"], shuffle=True, ve

In [3]:
name = "housing"                      # đổi thành "diabetes" hoặc "comments" (diabetes chạy lâu hơn)
D = dict(np.load("data/ch2_%s.npz" % name))
metrics, seconds, n_params, loss = C.run_keras(name, D, C.CFG[name], 42)
print(n_params, "tham số;", round(seconds, 1), "giây")
print({k: round(v, 4) for k, v in metrics.items()})

10369 tham số; 17.9 giây
{'r2_log': 0.6483, 'r2_price': 0.5546, 'mae': 1.6432, 'rmse': 4.0787, 'mape': 0.2828}
